In [3]:
pip install flwr tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.7/200.7 MB 10.4 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 9.4 MB/s eta 0:00:005 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 9.6 MB/s eta 0:00:009.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 9.6 MB/s eta 0:00:009.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [tensorflow]37m━━ 18/19 [tensorflow]]
Note: you may need to restart the kernel to use updated packages.


In [4]:
"""
05_prepare_federated_data.py
Partition users into federated clients
"""

import pandas as pd
import numpy as np
import os
import pickle

print("="*60)
print("PREPARING DATA FOR FEDERATED LEARNING")
print("="*60)

os.chdir('/Users/srivanur/Documents/fedpoi_thesis')

# ==========================================
# STEP 1: LOAD DATA
# ==========================================
print("\nStep 1: Loading preprocessed data...")

df = pd.read_csv('data/processed/foursquare_filtered.csv')
interaction_matrix = pd.read_csv('data/processed/interaction_matrix.csv', index_col=0)

print(f"✓ Total users: {interaction_matrix.shape[0]}")
print(f"✓ Total venues: {interaction_matrix.shape[1]}")

# ==========================================
# STEP 2: SPLIT USERS INTO CLIENTS
# ==========================================
print("\nStep 2: Partitioning users into federated clients...")

# Configuration
n_clients = 10  # Number of federated clients
print(f"Creating {n_clients} federated clients...")

# Get all user IDs
all_user_ids = interaction_matrix.index.values

# Shuffle for random assignment
np.random.seed(42)
np.random.shuffle(all_user_ids)

# Split users into n_clients groups
user_splits = np.array_split(all_user_ids, n_clients)

# Create client data
client_data = {}

for client_id in range(n_clients):
    client_users = user_splits[client_id]
    
    # Get interaction matrix for this client's users
    client_matrix = interaction_matrix.loc[client_users]
    
    # Get check-in data for these users
    client_checkins = df[df['user_id_encoded'].isin(client_users)]
    
    client_data[client_id] = {
        'user_ids': client_users,
        'matrix': client_matrix,
        'checkins': client_checkins,
        'n_users': len(client_users),
        'n_checkins': len(client_checkins)
    }
    
    print(f"Client {client_id}: {len(client_users):3d} users, {len(client_checkins):5d} check-ins")

# ==========================================
# STEP 3: TRAIN/TEST SPLIT FOR EACH CLIENT
# ==========================================
print("\nStep 3: Creating train/test splits for each client...")

for client_id in range(n_clients):
    matrix = client_data[client_id]['matrix'].values
    
    train_matrix = np.zeros_like(matrix)
    test_matrix = np.zeros_like(matrix)
    
    for user_idx in range(matrix.shape[0]):
        visited_venues = np.where(matrix[user_idx] == 1)[0]
        
        if len(visited_venues) >= 2:
            n_test = max(1, int(len(visited_venues) * 0.2))
            test_venues = np.random.choice(visited_venues, size=n_test, replace=False)
            train_venues = np.setdiff1d(visited_venues, test_venues)
            
            train_matrix[user_idx, train_venues] = 1
            test_matrix[user_idx, test_venues] = 1
        else:
            train_matrix[user_idx] = matrix[user_idx]
    
    client_data[client_id]['train_matrix'] = train_matrix
    client_data[client_id]['test_matrix'] = test_matrix
    
    print(f"Client {client_id}: Train={train_matrix.sum():.0f}, Test={test_matrix.sum():.0f}")

# ==========================================
# STEP 4: ANALYZE DATA DISTRIBUTION
# ==========================================
print("\nStep 4: Analyzing data distribution (Non-IID)...")

# Check how different each client's data is
venue_popularity_per_client = []

for client_id in range(n_clients):
    train = client_data[client_id]['train_matrix']
    venue_counts = train.sum(axis=0)  # How many times each venue is visited
    venue_popularity_per_client.append(venue_counts)

# Calculate similarity between clients (for thesis discussion)
print("\nData heterogeneity analysis:")
print("Top 10 venues per client (showing non-IID nature):")

for client_id in range(min(3, n_clients)):  # Show first 3 clients
    venue_counts = venue_popularity_per_client[client_id]
    top_venues = np.argsort(venue_counts)[-10:][::-1]
    print(f"\nClient {client_id} top venues: {top_venues[:5].tolist()}...")

print("\n✓ Data is non-IID (different clients have different popular venues)")
print("  This is realistic for geographic data!")

# ==========================================
# STEP 5: SAVE CLIENT DATA
# ==========================================
print("\nStep 5: Saving federated client data...")

# Create directory
os.makedirs('data/federated', exist_ok=True)

# Save each client's data
for client_id in range(n_clients):
    client_dir = f'data/federated/client_{client_id}'
    os.makedirs(client_dir, exist_ok=True)
    
    # Save matrices
    np.save(f'{client_dir}/train_matrix.npy', client_data[client_id]['train_matrix'])
    np.save(f'{client_dir}/test_matrix.npy', client_data[client_id]['test_matrix'])
    np.save(f'{client_dir}/user_ids.npy', client_data[client_id]['user_ids'])
    
    # Save metadata
    metadata = {
        'client_id': client_id,
        'n_users': client_data[client_id]['n_users'],
        'n_checkins': client_data[client_id]['n_checkins'],
        'n_train': client_data[client_id]['train_matrix'].sum(),
        'n_test': client_data[client_id]['test_matrix'].sum()
    }
    
    with open(f'{client_dir}/metadata.pkl', 'wb') as f:
        pickle.dump(metadata, f)

# Save overall configuration
config = {
    'n_clients': n_clients,
    'n_users_total': interaction_matrix.shape[0],
    'n_venues_total': interaction_matrix.shape[1],
    'random_seed': 42
}

with open('data/federated/config.pkl', 'wb') as f:
    pickle.dump(config, f)

print(f"✓ Saved data for {n_clients} clients to data/federated/")

# ==========================================
# STEP 6: CREATE SUMMARY
# ==========================================
print("\nStep 6: Creating summary report...")

with open('data/federated/summary.txt', 'w') as f:
    f.write("FEDERATED DATA PARTITION SUMMARY\n")
    f.write("="*60 + "\n\n")
    f.write(f"Total users: {interaction_matrix.shape[0]}\n")
    f.write(f"Total venues: {interaction_matrix.shape[1]}\n")
    f.write(f"Number of clients: {n_clients}\n\n")
    f.write("Client Distribution:\n")
    f.write("-"*60 + "\n")
    
    for client_id in range(n_clients):
        f.write(f"Client {client_id:2d}: ")
        f.write(f"{client_data[client_id]['n_users']:3d} users, ")
        f.write(f"{client_data[client_id]['n_checkins']:5d} check-ins, ")
        f.write(f"Train: {client_data[client_id]['train_matrix'].sum():.0f}, ")
        f.write(f"Test: {client_data[client_id]['test_matrix'].sum():.0f}\n")

print("✓ Saved summary to data/federated/summary.txt")

# ==========================================
# FINAL SUMMARY
# ==========================================
print("\n" + "="*60)
print("✓ FEDERATED DATA PREPARATION COMPLETE!")
print("="*60)
print(f"\nCreated {n_clients} federated clients")
print(f"Each client has private data (users + check-ins)")
print(f"Data is non-IID (realistic geographic distribution)")
print(f"\nNext step: Build federated learning system!")
print("="*60)

PREPARING DATA FOR FEDERATED LEARNING

Step 1: Loading preprocessed data...
✓ Total users: 1083
✓ Total venues: 38333

Step 2: Partitioning users into federated clients...
Creating 10 federated clients...
Client 0: 109 users, 23950 check-ins
Client 1: 109 users, 23060 check-ins
Client 2: 109 users, 23441 check-ins
Client 3: 108 users, 22959 check-ins
Client 4: 108 users, 22430 check-ins
Client 5: 108 users, 19643 check-ins
Client 6: 108 users, 23117 check-ins
Client 7: 108 users, 23467 check-ins
Client 8: 108 users, 22194 check-ins
Client 9: 108 users, 23167 check-ins

Step 3: Creating train/test splits for each client...
Client 0: Train=7629, Test=1854
Client 1: Train=7439, Test=1798
Client 2: Train=7225, Test=1756
Client 3: Train=8344, Test=2030
Client 4: Train=7312, Test=1779
Client 5: Train=7190, Test=1742
Client 6: Train=7673, Test=1861
Client 7: Train=6607, Test=1598
Client 8: Train=6822, Test=1659
Client 9: Train=7004, Test=1702

Step 4: Analyzing data distribution (Non-IID)...
